In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import glob

In [3]:
# Read the PMC IDs from file
file_path = 'pubmed_central_decompressed/example_pmc_ids'
with open(file_path, "r") as file:
    pmc_list = [line.strip() for line in file]

pmc_list = ['PMC5628943'] #+ pmc_list[:8]

cot_query = """
Extract the information of gene, DNA mutation, protein mutation, related disease, experiment result, and pathogenicity from this research paper paragraph in the format of ##gene, DNA mutation, protein mutation, disease, experimental result, pathogenicity. Follow these steps to ensure accuracy:

1. Identify Relevant Genes and Mutations, locate any gene names or symbols mentioned in the paper.
2. Identify the mutation as DNA-level or protein-level mutation, or relevant expression. Leave one of the mutation blank if only has either DNA or protein mutation.
3. Determine Disease Association: identify any diseases or conditions related to each gene and mutation. Look for terms like "disease," "syndrome," "condition," or names of specific conditions.
4. Extract Functional or Experimental Result: find statements describing the effect of each mutation, especially in terms of experimental assays or functional impacts. Note results like "rescued" or "failed to rescue" a function.
5. Assign Pathogenicity Level: determine if the mutation is "pathogenic," "benign," or similar terms indicating pathogenicity or clinical relevance.
6. Format Output Concisely: present the result in this format: ##gene, DNA mutation, protein mutation, disease, experimental result, pathogenicity

Using these steps, extract and present the unique ##gene, DNA mutation, protein mutation, disease, experiment result, pathogenicity for each mutation-disease relationship in this research paper only once. Stop when you find all mutations.
"""

In [16]:
def keep_paragraph(para):
    if len(para)>500:
        return True
    else:
        print('Num of words:', len(para.split()),'||', para,end='\n\n')
        return False

# Function to extract all paragraphs along with their sections and abstract
def get_full_text_paragraphs(i_pmc):
    directory_path = f"/mnt/isilon/wang_lab/shared/pubmed_central_decompressed/pmc_text/{i_pmc}/"
    nxml_files = glob.glob(f"{directory_path}*.nxml")

    if nxml_files:
        with open(nxml_files[0], "r", encoding="utf-8") as file:
            content = file.read()
        soup = BeautifulSoup(content, "xml")
        abstract = soup.find("abstract").get_text(separator=" ", strip=True) if soup.find("abstract") else ""
        
        paragraphs_with_sections = []
        for sec in soup.find_all("sec"):
            section_title = sec.find("title").get_text(strip=True) if sec.find("title") else "No Title"
            for p in sec.find_all("p"):
                paragraph_text = p.get_text(separator=" ", strip=True)
                if keep_paragraph(paragraph_text):
                    paragraphs_with_sections.append((section_title, paragraph_text))
                
                
        return abstract, paragraphs_with_sections
    return "", []


# Combine all inputs for inference
all_inputs = []
pmc_ids_for_inputs = []

for i_pmc in pmc_list:
    abstract, paragraphs_with_sections = get_full_text_paragraphs(i_pmc)
    full_text = ['Abstract:\n' + abstract + '\nTask:\n'] + \
                ['Paragraph:\n' + par + '\nTask:\n' for (sec, par) in paragraphs_with_sections]
    all_inputs.extend(full_text)
    pmc_ids_for_inputs.extend([i_pmc] * len(full_text))  # Keep track of PMCID for each input
    print(i_pmc,'Num of paragraphs:',len(full_text))


Num of words: 47 || (A-T) Craniofacial morphologies of maternal-null irf6 -/- embryos rescued by human IRF6 missense gene variant mRNA microinjections (100 pg/embryo) at 96 hpf stained with alcian blue. (A-D) Uninjected wild type control. (E-H) p.P12L. (I-L) p.P76L. (M-P) p.T100A. (Q-T) p.P222L. Scale bars = 150 μm, n = 3.

Num of words: 47 || (A-T) Craniofacial morphologies of maternal-null irf6 -/- embryos rescued by human IRF6 missense gene variant mRNA microinjections (100 pg/embryo) at 96 hpf stained with alcian blue. (A-D) Uninjected wild type control. (E-H) p.P12L. (I-L) p.P76L. (M-P) p.T100A. (Q-T) p.P222L. Scale bars = 150 μm, n = 3.

Num of words: 23 || All work with zebrafish (adult, larval, and embryonic) was performed in strict accordance with protocols approved by Massachusetts General Hospital IACUC (Protocol 2010N000106).

Num of words: 66 || Zebrafish Danio rerio were maintained in accordance with approved institutional protocols at Massachusetts General Hospital, as d

In [7]:
all_inputs[0]

'Abstract:\nLarge-scale sequencing efforts have captured a rapidly growing catalogue of genetic variations. However, the accurate establishment of gene variant pathogenicity remains a central challenge in translating personal genomics information to clinical decisions. Interferon Regulatory Factor 6 ( IRF6 ) gene variants are significant genetic contributors to orofacial clefts. Although approximately three hundred IRF6 gene variants have been documented, their effects on protein functions remain difficult to interpret. Here, we demonstrate the protein functions of human IRF6 missense gene variants could be rapidly assessed in detail by their abilities to rescue the irf6 -/- phenotype in zebrafish through variant mRNA microinjections at the one-cell stage. The results revealed many missense variants previously predicted by traditional statistical and computational tools to be loss-of-function and pathogenic retained partial or full protein function and rescued the zebrafish irf6 -/- pe

In [25]:
import re
from bs4 import BeautifulSoup
import glob

# Function to determine if a paragraph should be kept
def keep_paragraph(para):
    
    # Count the number of words
    words = para.split()

    # Rule 1: Word Count Threshold
    if len(words) > 10000:
        print('###rule 1 filter > 10000 words###\n', para)
        return False

    # Rule 2: Keyword-Based Filtering #not using 
    '''keywords = ["variant", "mutation", "snp", "polymorphism"]#, "allele", "substitution", 
                #"vus", "dna change", "protein change"]
    # Check for keywords in the paragraph
    matching_keywords = [keyword for keyword in keywords if keyword in para.lower()]
    if matching_keywords:
        #print('### Rule 2 Pass ###')
        #print('Matched Keywords:', matching_keywords)
        return False'''

    # Rule 3: Symbol-Based Filtering
    symbol_pattern = r'\b(?:c\.|p\.|rs\d+|[ATCG]>[ATCG]|[ATCG]->[ATCG]|[A-Z][a-z]+\d+[a-zA-Z*]+)\b'
    matched_terms = re.findall(symbol_pattern, para)
    if matched_terms:
        print('### Rule 3 Pass ###')
        print('Matched Terms:', matched_terms, '||', para)
        print(para)
        return True

    # Default: Exclude
    #print('###default filter NOT PASS###\n', para)
    return False


# Function to extract all paragraphs along with their sections and abstract
def get_full_text_paragraphs(i_pmc):
    directory_path = f"/mnt/isilon/wang_lab/shared/pubmed_central_decompressed/pmc_text/{i_pmc}/"
    nxml_files = glob.glob(f"{directory_path}*.nxml")

    if nxml_files:
        with open(nxml_files[0], "r", encoding="utf-8") as file:
            content = file.read()
        soup = BeautifulSoup(content, "xml")
        abstract = soup.find("abstract").get_text(separator=" ", strip=True) if soup.find("abstract") else ""
        
        paragraphs_with_sections = []
        for sec in soup.find_all("sec"):
            section_title = sec.find("title").get_text(strip=True) if sec.find("title") else "No Title"
            
            # Skip irrelevant sections based on title
            if section_title.lower() in ["methods", "method", "acknowledgments", "acknowledgment",
                                         "references", "reference"]:
                continue
            
            for p in sec.find_all("p"):
                paragraph_text = p.get_text(separator=" ", strip=True)
                if keep_paragraph(paragraph_text):
                    paragraphs_with_sections.append((section_title, paragraph_text))
                
        return abstract, paragraphs_with_sections
    return "", []


# Combine all inputs for inference
all_inputs = []
pmc_ids_for_inputs = []

for i_pmc in pmc_list:
    abstract, paragraphs_with_sections = get_full_text_paragraphs(i_pmc)
    full_text = ['Abstract:\n' + abstract + '\nTask:\n'] + \
                ['Paragraph:\n' + par + '\nTask:\n' for (sec, par) in paragraphs_with_sections]
    all_inputs.extend(full_text)
    pmc_ids_for_inputs.extend([i_pmc] * len(full_text))  # Keep track of PMCID for each input
    print(len(full_text))


### Rule 3 Pass ###
Matched Terms: ['c.'] || To establish a zebrafish irf6 null model and investigate the role of irf6 in periderm and craniofacial development, CRISPR-Cas9 was used for mutagenesis targeted at exon 6 of irf6 at the start of the SMIR/IAD protein-binding domain ( Fig 1A ). CRISPR-injected P0 embryos were raised to adulthood, out-crossed with wild type Tü adults, and genotyped at the gRNA target site. A F1 line was identified containing an 8 bp deletion in exon 6 of the irf6 coding region (NC_007133.7(NM_200598.2):c.772_779del), here forward referred to as “Δ8bp” ( Fig 1A ). Heterozygous F1 embryos ( irf6 +/Δ8bp ) were in-crossed to produce wild type, heterozygous and homozygous progeny at the expected Mendelian ratios with normal embryonic and craniofacial development ( Fig 1B ).
To establish a zebrafish irf6 null model and investigate the role of irf6 in periderm and craniofacial development, CRISPR-Cas9 was used for mutagenesis targeted at exon 6 of irf6 at the start o

In [120]:
df=pd.DataFrame([pmc_ids_for_inputs,all_inputs]).T

In [133]:
print(df[df[0]=='PMC10000012'][1])

27    Abstract:\nAs the world braces to enter its fo...
Name: 1, dtype: object


In [122]:
pd.DataFrame([pmc_ids_for_inputs,all_inputs]).T.groupby(0).count().sum()

1    43
dtype: int64

In [123]:
for i in range(len(all_inputs)):
    if 34<i<60:
        print(i)
        print(all_inputs[i])

35
Paragraph:
The Cow Comfort Index (CCI) was proposed by Nelson in 1996 [ 35 ] and is now widely used as a stall comfort assessment. In general, the maximum value of the comfort index for cows occurs in the morning and 1 h after returning from afternoon milking [ 36 ]; a CCI > 85% indicates high cow comfort, and this index can also be applied to buffaloes (Buffalo Comfort Index). In this study, both groups of buffalo BCI (a.m.) were greater than 85%, indicating high comfort level of buffalo. However, the BCI (p.m.) of the CB group was less than 85% in July–September, and the BCI (p.m.) of the FMB group was greater than 85%, indicating that fermented manure bedding can improve the comfort of buffalo. In summary, the application of FMB improved the lying behavior of buffaloes compared to CB bedding.
Task:

36
Paragraph:
The Cow Comfort Index (CCI) was proposed by Nelson in 1996 [ 35 ] and is now widely used as a stall comfort assessment. In general, the maximum value of the comfort inde

In [12]:
# Load the PMC IDs into a list (replace this with reading from your file)
pmc_ids = []  # Replace with the actual list of 2.6 million PMC IDs

# Example: Load PMC IDs from a text file (one ID per line)
with open('pubmed_central_decompressed/pmc_ids', 'r') as file:
    pmc_ids = file.read().splitlines()

# Batch size
batch_size = 10000

# Create batches
batches = []
for i in range(0, len(pmc_ids), batch_size):
    batch = pmc_ids[i:i+batch_size]
    batch_number = (i // batch_size) + 1
    batch_name = f"{batch_number}-{batch[0]}-{batch[-1]}"
    batches.append((batch_name, batch))

# Write each batch to its own file
for batch_name, batch in batches:
    with open(f"pubmed_central_decompressed/batch_pmc_ids/{batch_name}.txt", 'w') as file:
        file.write("\n".join(batch))

print(f"Generated {len(batches)} batches.")


Generated 262 batches.


In [22]:
import os
import re
from bs4 import BeautifulSoup
import glob
import pandas as pd

# Function to determine if a paragraph should be kept
def keep_paragraph(para):
    # Count the number of words
    words = para.split()

    # Rule 1: Word Count Threshold
    if len(words) < 30:
        return False

    # Rule 2: Keyword-Based Filtering (Optional, uncomment if needed)
    '''keywords = ["variant", "mutation", "gene", "snp", "polymorphism", "vus", "dna change", "protein change"]
    matching_keywords = [keyword for keyword in keywords if keyword in para.lower()]
    if matching_keywords:
        return True'''

    # Rule 3: Symbol-Based Filtering
    symbol_pattern = r'\b(?:c\.|p\.|rs\d+|[ATCG]>[ATCG]|[ATCG]->[ATCG]|[A-Z][a-z]+\d+[a-zA-Z*]+)\b'
    if re.search(symbol_pattern, para):
        return True

    # Default: Exclude
    return False

# Function to extract all paragraphs along with their sections and abstract
def get_full_text_paragraphs(i_pmc):
    directory_path = f"/mnt/isilon/wang_lab/shared/pubmed_central_decompressed/pmc_text/{i_pmc}/"
    nxml_files = glob.glob(f"{directory_path}*.nxml")

    if nxml_files:
        with open(nxml_files[0], "r", encoding="utf-8") as file:
            content = file.read()
        soup = BeautifulSoup(content, "xml")
        
        # Extract abstract
        abstract = soup.find("abstract").get_text(separator=" ", strip=True) if soup.find("abstract") else ""

        # Extract paragraphs
        paragraphs = []
        for sec in soup.find_all("sec"):
            section_title = sec.find("title").get_text(strip=True) if sec.find("title") else "No Title"

            # Skip irrelevant sections
            if section_title.lower() in ["methods", "method", "acknowledgments", "acknowledgment", "references", "reference"]:
                continue

            for p in sec.find_all("p"):
                paragraph_text = p.get_text(separator=" ", strip=True)
                if keep_paragraph(paragraph_text):
                    paragraphs.append(paragraph_text)

        return abstract, paragraphs
    return "", []

# Process a batch of PMC IDs from a file
def process_batch_file(batch_file, output_directory):
    # Read PMC IDs from the batch file
    with open(batch_file, "r") as f:
        pmc_ids = f.read().splitlines()

    rows = []

    for pmc_id in pmc_ids:
        abstract, paragraphs = get_full_text_paragraphs(pmc_id)
        
        # Add abstract as a row
        if abstract:
            rows.append([pmc_id, abstract])

        # Add filtered paragraphs
        for paragraph in paragraphs:
            rows.append([pmc_id, paragraph])

    # Save the processed batch to a new CSV file in the output directory
    batch_name = os.path.basename(batch_file).replace(".txt", ".csv")
    output_file = os.path.join(output_directory, batch_name)
    df = pd.DataFrame(rows, columns=["PMC_ID", "Paragraph"])
    df.to_csv(output_file, index=False)
    print(f"Processed batch saved to {output_file}")

# Input and output directories
batch_directory = "pubmed_central_decompressed/batch_pmc_ids/"  # Adjust the path to your batch files
output_directory = "pubmed_central_decompressed/batch_filtered_input/"  # Path to save filtered results

# Create output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

batch_files = set(glob.glob(os.path.join(batch_directory, "*.txt")))
finished_files = glob.glob(os.path.join(output_directory, "*.csv"))
finished_files = set([file.replace('.csv','.txt').replace('batch_filtered_input','batch_pmc_ids') for file in finished_files])
batch_files=batch_files-finished_files
for batch_file in batch_files:
    process_batch_file(batch_file, output_directory)

Processed batch saved to pubmed_central_decompressed/batch_filtered_input/107-PMC5889246-PMC5918865.csv
Processed batch saved to pubmed_central_decompressed/batch_filtered_input/51-PMC4139646-PMC4175892.csv
Processed batch saved to pubmed_central_decompressed/batch_filtered_input/25-PMC3127301-PMC3173977.csv
Processed batch saved to pubmed_central_decompressed/batch_filtered_input/238-PMC9412827-PMC9435967.csv
Processed batch saved to pubmed_central_decompressed/batch_filtered_input/200-PMC8493111-PMC8518438.csv
Processed batch saved to pubmed_central_decompressed/batch_filtered_input/56-PMC4316728-PMC4351842.csv
Processed batch saved to pubmed_central_decompressed/batch_filtered_input/192-PMC8287318-PMC8310308.csv
Processed batch saved to pubmed_central_decompressed/batch_filtered_input/147-PMC7087806-PMC7117321.csv
Processed batch saved to pubmed_central_decompressed/batch_filtered_input/90-PMC5388239-PMC5416873.csv
Processed batch saved to pubmed_central_decompressed/batch_filtered_

KeyboardInterrupt: 

In [20]:
# Input and output directories
batch_directory = "pubmed_central_decompressed/batch_pmc_ids/"  # Adjust the path to your batch files
output_directory = "pubmed_central_decompressed/batch_filtered_input/"  # Path to save filtered results

# Create output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

batch_files = set(glob.glob(os.path.join(batch_directory, "*.txt")))
finished_files = glob.glob(os.path.join(output_directory, "*.csv"))
finished_files = set([file.replace('.csv','.txt').replace('batch_filtered_input','batch_pmc_ids') for file in finished_files])
batch_files=batch_files-finished_files

In [16]:
len(finished_files)

71

In [21]:
len(batch_files)

191

In [19]:
len(batch_files-finished_files)

191